# Creación del Modelo ML

## Características y suposiciones

### Segmentación de muestras (entrenamiento y test)

No podemos usar el train_test_split con la configuración por defecto porque mezcla filas al azar. Como nuestro dataset tiene un sentido temporal, podríamos estar entrenando el modelo con datos de cansancio de un jugador en marzon para predecir si se lesionó en noviembre. 

Para solucionarlo ordenaremos el DataFrame por fecha y haremos el corte manualmente, seleccionando los primeros meses como conjunto de entrenamiento y los últimos meses como conjunto de test, para ver si el modelo es capaz de predecir el futuro. 

### Desbalance de clases

Nuestro dataset tiene miles de filas pero muy pocas lesiones (en torno al 95% de los datos son 0s). Si le pasamos esto a un modelo normal, el modelo predecirá siempre 0 con el objetivo de lograr la máxima precisión posible. Para nuestro entrenador, un modelo que nunca le avise de una lesión es bastante inútil. 

Para solucionarlo usaremos un parámetros llamado class_weight='balanced', que nos permitirá ponderar los resultados y otorgar una puntuación al modelo si consigue predecir correctamente una lesión (1).

### Elección del modelo

Vamos a usar un RandomForestClassifier porque nuestro dataset sigue una relación no lineal: el riesgo de lesión sube si el valor de ACWR es muy bajo y si es muy alto (forma de U.)

Además no queremos decirle al entrenador "El jugador se va a lesionar hoy" sino más bien darle la probabilidad de que se lesione "Tiene un 65% de riesgo de lesión asi que usaremos predict_proba(). 

In [5]:
from consulta_sql import df_ml

In [13]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier


## Preparación de los datos

In [6]:
df_ml.head()

,ID_Jugador,Fecha,Lesion,carga_aguda,carga_cronica,acwr,media_sueno_3d,es_visitante,alerta_sueno
0,JUG_01,2025-09-28,0.0,335.87,322.81,1.04,8.43,1,0
1,JUG_02,2025-09-28,0.0,72.00,89.44,0.81,7.80,1,0
2,JUG_03,2025-09-28,0.0,445.43,380.36,1.17,7.90,1,0
3,JUG_04,2025-09-28,0.0,303.36,353.55,0.86,7.07,1,0
4,JUG_05,2025-09-28,0.0,356.53,353.64,1.01,8.67,0,0


### Creación de matrices y separación de muestras

In [7]:
# Ordenar por fecha
df_ml = df_ml.sort_values("Fecha")

#  Separación de predictores y target
X = df_ml[["carga_aguda", "carga_cronica", "acwr", "media_sueno_3d", "es_visitante", "alerta_sueno"]]
y = df_ml[["Lesion"]]

In [8]:
# Definición del índice de corte
indice_corte = int(len(df_ml) * 0.8)

# Separación de matrices
# Datos del pasado (Para entrenar)
X_train = X.iloc[:indice_corte]
y_train = y.iloc[:indice_corte]

# Datos del futuro (Para examinar al modelo)
X_test = X.iloc[indice_corte:]
y_test = y.iloc[indice_corte:]

## Creación del modelo

In [14]:
# 1. Crear el modelo 
modelo_rf = RandomForestClassifier(
    n_estimators=100, 
    class_weight='balanced', 
    random_state=42 # Reproducibilidad
)

# 2. Entrenar el modelo
modelo_rf.fit(X_train, y_train)

# 3. Testear en datos nuevos (meses futuros)
probabilidades_totales = modelo_rf.predict_proba(X_test)

# 4. Nos quedamos con la columna de riesgo de lesión
riesgo_lesion_test = probabilidades_totales[:, 1]

# 5. Mostramos resultados
print("Primeras 5 predicciones de riesgo de lesión:", riesgo_lesion_test[:5])

Primeras 5 predicciones de riesgo de lesión: [0.03 0.   0.   0.   0.  ]


c:\Users\ronge\Desktop\PROGRAMACION\DATA SCIENCE\Proyectos SQL\Analisis Lesiones\venv\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


## Evaluación del modelo


(2556, 6)

(639, 6)

df_ml